In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [3]:
print(spark.version)

3.5.8


In [4]:
!curl -L -o yellow_tripdata_2025-11.parquet https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  1 67.8M    1 1333k    0     0  1249k      0  0:00:55  0:00:01  0:00:54 1252k
 25 67.8M   25 17.2M    0     0  8535k      0  0:00:08  0:00:02  0:00:06 8544k
 49 67.8M   49 33.6M    0     0  10.9M      0  0:00:06  0:00:03  0:00:03 10.9M
 74 67.8M   74 50.2M    0     0  12.3M      0  0:00:05  0:00:04  0:00:01 12.3M
 98 67.8M   98 66.6M    0     0  13.1M      0  0:00:05  0:00:05 --:--:-- 13.3M
100 67.8M  100 67.8M    0     0  13.2M      0  0:00:05  0:00:05 --:--:-- 16.3M


In [5]:
df = spark.read.parquet('yellow_tripdata_2025-11.parquet')

In [6]:
df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [7]:
df.dtypes

[('VendorID', 'int'),
 ('tpep_pickup_datetime', 'timestamp_ntz'),
 ('tpep_dropoff_datetime', 'timestamp_ntz'),
 ('passenger_count', 'bigint'),
 ('trip_distance', 'double'),
 ('RatecodeID', 'bigint'),
 ('store_and_fwd_flag', 'string'),
 ('PULocationID', 'int'),
 ('DOLocationID', 'int'),
 ('payment_type', 'bigint'),
 ('fare_amount', 'double'),
 ('extra', 'double'),
 ('mta_tax', 'double'),
 ('tip_amount', 'double'),
 ('tolls_amount', 'double'),
 ('improvement_surcharge', 'double'),
 ('total_amount', 'double'),
 ('congestion_surcharge', 'double'),
 ('Airport_fee', 'double'),
 ('cbd_congestion_fee', 'double')]

In [8]:
df = df.repartition(4)

In [9]:
df.write.parquet('data/raw/yellow/2025/11/')

### Question 2: Yellow November 2025

![Spark result](question_2_answer.png)

#### Answer: The average size of the Parquet Files that were created is 25MB

In [10]:
df_yellow_nov_2025 = spark.read.parquet('data/raw/yellow/2025/11/')
df_yellow_nov_2025.registerTempTable('yellow_nov_2025')

C:\Users\adilb\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyspark\sql\dataframe.py:329: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


### Question 3: Count records

In [13]:
df_count_records = spark.sql("""
SELECT 
    COUNT(*) AS taxi_trips_num 
FROM
    yellow_nov_2025
WHERE
    DATE(tpep_pickup_datetime) = '2025-11-15'
""")

In [14]:
df_count_records.show()

+--------------+
|taxi_trips_num|
+--------------+
|        162604|
+--------------+



### Question 4: Longest trip

In [30]:
df_longest_trip = spark.sql("""
SELECT 
    trip_distance,
    MAX(timestampdiff(
        MINUTE,
        tpep_pickup_datetime,
        tpep_dropoff_datetime
    ) / 60.0) AS max_trip_duration_hours

FROM
    yellow_nov_2025

GROUP BY trip_distance
ORDER BY max_trip_duration_hours DESC
LIMIT 3;

""")

In [31]:
df_longest_trip.show()

+-------------+-----------------------+
|trip_distance|max_trip_duration_hours|
+-------------+-----------------------+
|       121.17|              90.633333|
|         1.08|              76.933333|
|          0.0|              76.200000|
+-------------+-----------------------+



### Question 6:  Least frequent pickup location zone

In [32]:
df_zones = spark.read.parquet('zones/')

In [33]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [35]:
df_result = df_yellow_nov_2025.join(df_zones, df_yellow_nov_2025.PULocationID == df_zones.LocationID)

In [37]:
df_result.drop('LocationID').show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+---------+--------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|  Borough|                Zone|service_zone|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+---------+--------------------+------------

In [38]:
df_result.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- LocationID: string (nullable = true)
 |-- Borough: string (nullable = t

In [40]:
df_result.registerTempTable('yellow_taxi_zone')

In [46]:
df_yellow_locatio0n_zone = spark.sql("""
SELECT 
    Zone,
    COUNT(*) AS pickup_count

FROM
    yellow_taxi_zone
GROUP BY
    Zone
ORDER BY
    pickup_count ASC
LIMIT 5;
""")

In [47]:
df_yellow_locatio0n_zone.show()

+--------------------+------------+
|                Zone|pickup_count|
+--------------------+------------+
|Governor's Island...|           1|
|Eltingville/Annad...|           1|
|       Arden Heights|           1|
|       Port Richmond|           3|
|       Rikers Island|           4|
+--------------------+------------+

